In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make `src/` importable from the notebooks/ directory.
sys.path.insert(0, str(Path.cwd().parent))
from src.mappings import (
    LOCATION_MAPPING,
    AWT_MAPPING,
    TURF_MAPPING,
    CLASS_MAPPING,
    UNKNOWN_CLASS_VALUE,
)

# Seasons <= TRAIN_SEASON_END are used to compute imputation statistics
# so valid/test seasons cannot leak into the medians.
TRAIN_SEASON_END = 14

In [2]:
df = pd.read_csv('../data/HKHJ_Dataset_Feature_Engineered.csv')

In [3]:
df.shape

(158563, 60)

In [4]:
df.head()

,date,location,race_no,info,going,course,place,horse_number,horse_name,horse_name_id,...,performance_within_race_trend,performance_within_race_trend_confidence,performance_within_conditions_trend,performance_within_conditions_trend_confidence,weight_trend,weight_trend_confidence,weight_trend_direction,on_date_horse_weight_trend,on_date_horse_weight_trend_confidence,on_date_horse_weight_trend_direction
0,2009-01-01 00:00:00+08:00,Sha Tin,RACE 1 (264),Class 4 - 1600M - (60-40),GOOD TO FIRM,"TURF - ""B+2"" Course",1,1.0,BEAUTIFUL CHOICE(CH130),CH130,...,NaN,0.0,NaN,0.0,NaN,0.2,NaN,NaN,0.2,NaN
1,2009-01-01 00:00:00+08:00,Sha Tin,RACE 1 (264),Class 4 - 1600M - (60-40),GOOD TO FIRM,"TURF - ""B+2"" Course",2,9.0,LIVERBIRD(CE055),CE055,...,NaN,0.0,NaN,0.0,NaN,0.2,NaN,NaN,0.2,NaN
2,2009-01-01 00:00:00+08:00,Sha Tin,RACE 1 (264),Class 4 - 1600M - (60-40),GOOD TO FIRM,"TURF - ""B+2"" Course",3,10.0,GREEN GLORY(CJ177),CJ177,...,NaN,0.0,NaN,0.0,NaN,0.2,NaN,NaN,0.2,NaN
3,2009-01-01 00:00:00+08:00,Sha Tin,RACE 1 (264),Class 4 - 1600M - (60-40),GOOD TO FIRM,"TURF - ""B+2"" Course",4,4.0,BRILLIANT FLASH(CE042),CE042,...,NaN,0.0,NaN,0.0,NaN,0.2,NaN,NaN,0.2,NaN
4,2009-01-01 00:00:00+08:00,Sha Tin,RACE 1 (264),Class 4 - 1600M - (60-40),GOOD TO FIRM,"TURF - ""B+2"" Course",5,2.0,DR WELL(CD165),CD165,...,NaN,0.0,NaN,0.0,NaN,0.2,NaN,NaN,0.2,NaN


In [5]:
df.isna().sum()

date                                                  0
location                                              0
race_no                                               0
info                                                  0
going                                                 0
course                                                0
place                                                 0
horse_number                                       1608
horse_name                                            0
horse_name_id                                         0
jockey                                                0
trainer                                               0
weight                                                0
on_date_horse_weight                               2967
draw                                               2958
length_behind_winner                               3399
running_position                                      0
finish_time                                     

### Variables that should be retained
- date
- location
- going
- place
- horse_number
- horse_name_id (create a dict here)
- jockey
- trainer
- weight
- on_date_horse_weight
- draw
- info_class
- info_distance
- info_rating
- surface_type
- rail_offset
- unique_id
- season
- races_in_season
- days_since_last_race_any
- days_since_last_race_same_season
- performance_within_race_rolling_mean_5
- performance_within_race_rolling_median_5
- performance_within_race_ewa_5
- performance_within_conditions_rolling_mean_5
- performance_within_conditions_rolling_median_5
- performance_within_conditions_rolling_count_5
- performance_within_conditions_ewa_5
- place_numeric_rolling_mean_5
- place_numeric_rolling_median_5
- place_numeric_rolling_count_5
- place_numeric_ewa_5
- performance_within_race_trend
- performance_within_race_trend_confidence
- performance_within_conditions_trend
- performance_within_conditions_trend_confidence
- on_date_horse_weight_trend
- on_date_horse_weight_trend_confidence
- on_date_horse_weight_trend_direction
- weight_trend
- weight_trend_confidence
- weight_trend_direction
- top3
- win

Next, we create a new dataframe with these columns

In [6]:
ml_columns = [
"date",
"location",
"going",
"horse_number",
"horse_name_id",
"jockey",
"trainer",
"weight",
"on_date_horse_weight",
"draw",
"info_class",
"info_distance",
"info_rating",  
"surface_type",
"rail_offset",
"unique_id",
"season",
"races_in_season",
"days_since_last_race_any",
"days_since_last_race_same_season",
"performance_within_race_rolling_mean_5",
"performance_within_race_rolling_median_5",
"performance_within_race_ewa_5",
"performance_within_conditions_rolling_mean_5",
"performance_within_conditions_rolling_median_5",
"performance_within_conditions_rolling_count_5",
"performance_within_conditions_ewa_5",
"place_numeric_rolling_mean_5",
"place_numeric_rolling_median_5",
"place_numeric_rolling_count_5",
"place_numeric_ewa_5",
"performance_within_race_trend",
"performance_within_race_trend_confidence",
"performance_within_conditions_trend",
"performance_within_conditions_trend_confidence",
"on_date_horse_weight_trend",
"on_date_horse_weight_trend_confidence",
"on_date_horse_weight_trend_direction",
"weight_trend",
"weight_trend_confidence",
"weight_trend_direction",
"top3",
"win"
]

In [7]:
df_new = df[ml_columns].copy()

In [8]:
df_new.head()

,date,location,going,horse_number,horse_name_id,jockey,trainer,weight,on_date_horse_weight,draw,...,performance_within_conditions_trend,performance_within_conditions_trend_confidence,on_date_horse_weight_trend,on_date_horse_weight_trend_confidence,on_date_horse_weight_trend_direction,weight_trend,weight_trend_confidence,weight_trend_direction,top3,win
0,2009-01-01 00:00:00+08:00,Sha Tin,GOOD TO FIRM,1.0,CH130,K C Leung,Y S Tsui,119,1114.0,4.0,...,NaN,0.0,NaN,0.2,NaN,NaN,0.2,NaN,1,1
1,2009-01-01 00:00:00+08:00,Sha Tin,GOOD TO FIRM,9.0,CE055,H W Lai,C S Shum,118,1125.0,8.0,...,NaN,0.0,NaN,0.2,NaN,NaN,0.2,NaN,1,0
2,2009-01-01 00:00:00+08:00,Sha Tin,GOOD TO FIRM,10.0,CJ177,W C Marwing,P F Yiu,118,1063.0,2.0,...,NaN,0.0,NaN,0.2,NaN,NaN,0.2,NaN,1,0
3,2009-01-01 00:00:00+08:00,Sha Tin,GOOD TO FIRM,4.0,CE042,O Doleuze,C Fownes,127,1144.0,9.0,...,NaN,0.0,NaN,0.2,NaN,NaN,0.2,NaN,0,0
4,2009-01-01 00:00:00+08:00,Sha Tin,GOOD TO FIRM,2.0,CD165,M L Yeung,C W Chang,118,1122.0,6.0,...,NaN,0.0,NaN,0.2,NaN,NaN,0.2,NaN,0,0


In [9]:
for col in df_new.select_dtypes(include=['object']).columns:
    print(f"Column: {col}")

Column: date
Column: location
Column: going
Column: horse_name_id
Column: jockey
Column: trainer
Column: info_class
Column: info_rating
Column: surface_type


/var/folders/ry/3xn1jktn0j33gx31vxb5brvr0000gn/T/ipykernel_12818/3297499916.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_new.select_dtypes(include=['object']).columns:


In [10]:
df_new['location'].unique()

<ArrowStringArray>
['Sha Tin', 'Happy Valley']
Length: 2, dtype: str

In [11]:
df_new['going'].unique()

<ArrowStringArray>
[    'GOOD TO FIRM',             'FAST',             'GOOD',
 'GOOD TO YIELDING',         'YIELDING',             'SOFT',
         'WET SLOW', 'YIELDING TO SOFT',             'SLOW',
         'WET FAST',            'HEAVY',           'SEALED']
Length: 12, dtype: str

In [12]:
df_new['info_class'].unique()

<ArrowStringArray>
[                    'Class 4',                     'Class 5',
                     'Class 3',                     'Class 2',
       'Hong Kong Group Three',                     'Class 1',
 'Class 4 (Special Condition)',         'Hong Kong Group One',
 'Class 4 (Bonus Prize Money)', 'Class 2 (Bonus Prize Money)',
 'Class 3 (Bonus Prize Money)',                'Griffin Race',
         'Hong Kong Group Two',             'Restricted Race',
 'Class 3 (Special Condition)',                   'Group One',
                   'Group Two',        'Class 4 (Restricted)',
                 'Group Three',                 '4 Year Olds',
        'Class 3 (Restricted)']
Length: 21, dtype: str

In [13]:
df_new['info_rating'].unique()

<ArrowStringArray>
[  '60-40',   '40-10',    '40-0',   '80-60',   '95-75',       nan,   '85-60',
  '100-80',  '110-90',   '90-70',   '65-40',  '110-85',  '105-80',  '115-90',
  '100-75',  '115-95',   '75-55',   '40-15',  '120-95',   '95-70',  '105-85',
 '120-100',   '40-20',   '60-35',   '80-55',  '110-80',   '95-80',  '100-70',
  '105-75',   '90-65',   '90-60']
Length: 31, dtype: str

In [14]:
df_new['surface_type'].unique()

<ArrowStringArray>
['TURF', 'AWT']
Length: 2, dtype: str

## Mapping Surface Type and Going

In [15]:
df_new[df_new['surface_type'] == 'AWT']['going'].unique()

<ArrowStringArray>
['FAST', 'GOOD', 'WET SLOW', 'SLOW', 'WET FAST', 'SEALED']
Length: 6, dtype: str

In [16]:
df_new[df_new['surface_type'] == 'TURF']['going'].unique()

<ArrowStringArray>
[    'GOOD TO FIRM',             'GOOD', 'GOOD TO YIELDING',
         'YIELDING',             'SOFT', 'YIELDING TO SOFT',
            'HEAVY']
Length: 7, dtype: str

In [17]:
df_new['surface_type_mapped'] = (df_new['surface_type'].str.upper() == 'AWT').astype('int8')

In [18]:
df_new['going_numeric'] = np.where(
    df_new['surface_type_mapped'] == 1,
    df_new['going'].map(AWT_MAPPING),
    df_new['going'].map(TURF_MAPPING),
)

In [19]:
df_new.head()

,date,location,going,horse_number,horse_name_id,jockey,trainer,weight,on_date_horse_weight,draw,...,on_date_horse_weight_trend,on_date_horse_weight_trend_confidence,on_date_horse_weight_trend_direction,weight_trend,weight_trend_confidence,weight_trend_direction,top3,win,surface_type_mapped,going_numeric
0,2009-01-01 00:00:00+08:00,Sha Tin,GOOD TO FIRM,1.0,CH130,K C Leung,Y S Tsui,119,1114.0,4.0,...,NaN,0.2,NaN,NaN,0.2,NaN,1,1,0,0.0
1,2009-01-01 00:00:00+08:00,Sha Tin,GOOD TO FIRM,9.0,CE055,H W Lai,C S Shum,118,1125.0,8.0,...,NaN,0.2,NaN,NaN,0.2,NaN,1,0,0,0.0
2,2009-01-01 00:00:00+08:00,Sha Tin,GOOD TO FIRM,10.0,CJ177,W C Marwing,P F Yiu,118,1063.0,2.0,...,NaN,0.2,NaN,NaN,0.2,NaN,1,0,0,0.0
3,2009-01-01 00:00:00+08:00,Sha Tin,GOOD TO FIRM,4.0,CE042,O Doleuze,C Fownes,127,1144.0,9.0,...,NaN,0.2,NaN,NaN,0.2,NaN,0,0,0,0.0
4,2009-01-01 00:00:00+08:00,Sha Tin,GOOD TO FIRM,2.0,CD165,M L Yeung,C W Chang,118,1122.0,6.0,...,NaN,0.2,NaN,NaN,0.2,NaN,0,0,0,0.0


## Encoding Horse Name, Jockey and Trainer

In [20]:
new_df = df_new.copy()
new_df['date'] = pd.to_datetime(new_df['date'])

# Stable order so target encoding is reproducible and races on the same day
# cannot leak into each other.
new_df = (
    new_df.sort_values(['date', 'unique_id', 'horse_number'])
          .reset_index(drop=True)
)

id_cols = ['horse_name_id', 'jockey', 'trainer']
target_cols = ['win', 'top3']
# Smoothing prior for target encoding. Higher = more shrinkage toward the
# global mean for rare IDs (e.g. a horse with only a handful of starts).
prior_weight = 5

for col in id_cols:
    # Frequency / experience encoding (log of total appearances).
    freq = new_df.groupby(col)[col].transform('count')
    new_df[f'{col}_freq'] = np.log1p(freq)

    # Expanding (leak-free) target encoding: every row only sees outcomes
    # from earlier rows for the same id.
    for target in target_cols:
        global_mean = new_df[target].mean()
        grp = new_df.groupby(col)[target]

        past_sum = grp.cumsum() - new_df[target]
        past_cnt = grp.cumcount()

        new_df[f'{col}_{target}_te'] = (
            past_sum + prior_weight * global_mean
        ) / (past_cnt + prior_weight)

## Mapping location

In [21]:
new_df['location_mapped'] = new_df['location'].map(LOCATION_MAPPING)

## Info Class Mapping

In [22]:
new_df['info_class_mapped'] = (
    new_df['info_class']
    .map(CLASS_MAPPING)
    .fillna(UNKNOWN_CLASS_VALUE)  # sentinel weaker than Class 5
)

## Info Rating Mapping

In [23]:
# Split "high-low" rating string into numeric high/low/mid columns.
# NaNs are preserved so the train-only median imputation below can handle them.
rating_split = (
    new_df['info_rating']
    .str.split('-', expand=True)
    .astype(float)
)

new_df['info_rating_high'] = rating_split[0]
new_df['info_rating_low']  = rating_split[1]
new_df['info_rating_mid']  = rating_split.mean(axis=1)

## Race-aware Features

Pferderennen sind ein Vergleich der Pferde *innerhalb eines Rennens*. Absolute
Werte wie `weight` oder `draw` sagen wenig — der Rang innerhalb des Feldes
ist viel aussagekräftiger.

In [24]:
race_grp = new_df.groupby('unique_id')

new_df['field_size'] = race_grp['horse_number'].transform('count')

# Percentile rank within race (0..1). NaNs in the source column propagate.
# `info_rating_mid` is excluded: it's the race-level rating band (e.g. "60-40"),
# constant within a race, so ranking it gives 0.5 for every horse — no signal.
for col in ['weight', 'on_date_horse_weight', 'draw']:
    new_df[f'{col}_rank_in_race'] = race_grp[col].rank(pct=True)

# Z-score within race for the rolling form features — captures whether a
# horse is above/below the field-average for that race.
for col in [
    'performance_within_race_ewa_5',
    'place_numeric_ewa_5',
    'horse_name_id_top3_te',
    'horse_name_id_win_te',
]:
    mean = race_grp[col].transform('mean')
    std = race_grp[col].transform('std').replace(0, np.nan)
    new_df[f'{col}_z_in_race'] = (new_df[col] - mean) / std

## Handling Missing Values

In [25]:
# Flag every column where NaN carries the meaning "no history available"
# so the model can learn from the missingness pattern itself.
history_cols = [
    'performance_within_conditions_trend', 'performance_within_race_trend',
    'on_date_horse_weight_trend', 'on_date_horse_weight_trend_direction',
    'performance_within_race_rolling_mean_5', 'performance_within_race_rolling_median_5',
    'performance_within_race_ewa_5', 'performance_within_conditions_rolling_mean_5',
    'performance_within_conditions_rolling_median_5', 'performance_within_conditions_ewa_5',
    'place_numeric_rolling_mean_5', 'place_numeric_rolling_median_5', 'place_numeric_ewa_5',
    'weight_trend', 'weight_trend_direction', 'days_since_last_race_any',
    'days_since_last_race_same_season', 'info_rating_high', 'info_rating_low', 'info_rating_mid',
    'on_date_horse_weight', 'draw', 'horse_number',
    # race-aware features inherit NaNs from their source columns
    'weight_rank_in_race', 'on_date_horse_weight_rank_in_race',
    'draw_rank_in_race',
    'performance_within_race_ewa_5_z_in_race', 'place_numeric_ewa_5_z_in_race',
    'horse_name_id_top3_te_z_in_race', 'horse_name_id_win_te_z_in_race',
]
for col in history_cols:
    new_df[f'{col}_missing'] = new_df[col].isna().astype('int8')

### Train-only imputation statistics

Medians werden ausschließlich aus den Trainings-Saisons berechnet
(`season <= TRAIN_SEASON_END`), um Look-ahead-Bias zu vermeiden. Werte aus
Validation- und Test-Saisons dürfen die Imputation nicht beeinflussen.

In [26]:
train_mask = new_df['season'] <= TRAIN_SEASON_END
train_view = new_df.loc[train_mask]
print(f"Train rows: {train_mask.sum():,} / {len(new_df):,}")

Train rows: 138,183 / 158,563


In [27]:
trend_cols = [
    'performance_within_conditions_trend', 'performance_within_race_trend',
    'on_date_horse_weight_trend', 'weight_trend'
]
new_df[trend_cols] = new_df[trend_cols].fillna(0.0)

In [28]:
rolling_cols = [
    'performance_within_race_rolling_mean_5', 'performance_within_race_rolling_median_5',
    'performance_within_race_ewa_5', 'performance_within_conditions_rolling_mean_5',
    'performance_within_conditions_rolling_median_5', 'performance_within_conditions_ewa_5',
    'place_numeric_rolling_mean_5', 'place_numeric_rolling_median_5', 'place_numeric_ewa_5',
    # race-aware features
    'weight_rank_in_race', 'on_date_horse_weight_rank_in_race',
    'draw_rank_in_race',
    'performance_within_race_ewa_5_z_in_race', 'place_numeric_ewa_5_z_in_race',
    'horse_name_id_top3_te_z_in_race', 'horse_name_id_win_te_z_in_race',
]
rolling_medians = train_view[rolling_cols].median()
new_df[rolling_cols] = new_df[rolling_cols].fillna(rolling_medians)

In [29]:
new_df['days_since_last_race_any'] = new_df['days_since_last_race_any'].fillna(999)
new_df['days_since_last_race_same_season'] = new_df['days_since_last_race_same_season'].fillna(999)

In [30]:
rating_cols = ['info_rating_high', 'info_rating_low', 'info_rating_mid']

# Map each row to the train-set median for its info_class_mapped, then fall
# back to the global train median for classes that don't appear in train.
class_medians = train_view.groupby('info_class_mapped')[rating_cols].median()
global_medians = train_view[rating_cols].median()

for col in rating_cols:
    fill_by_class = new_df['info_class_mapped'].map(class_medians[col])
    new_df[col] = new_df[col].fillna(fill_by_class).fillna(global_medians[col])

In [31]:
measure_cols = ['on_date_horse_weight', 'draw', 'horse_number']
measure_medians = train_view[measure_cols].median()
new_df[measure_cols] = new_df[measure_cols].fillna(measure_medians)

In [32]:
new_df.isna().sum()

date                                               0
location                                           0
going                                              0
horse_number                                       0
horse_name_id                                      0
                                                  ..
draw_rank_in_race_missing                          0
performance_within_race_ewa_5_z_in_race_missing    0
place_numeric_ewa_5_z_in_race_missing              0
horse_name_id_top3_te_z_in_race_missing            0
horse_name_id_win_te_z_in_race_missing             0
Length: 97, dtype: int64

In [33]:
direction_cols = ['on_date_horse_weight_trend_direction', 'weight_trend_direction']
new_df[direction_cols] = new_df[direction_cols].fillna(0)

In [34]:
output_path = Path('../data/HKHJ_Dataset_After_MV.parquet')
new_df.to_parquet(output_path, index=False)
print(f"Wrote {output_path} ({output_path.stat().st_size / 1e6:.1f} MB)")

Wrote ../data/HKHJ_Dataset_After_MV.parquet (25.5 MB)
